# 03 GARCH Modeling - 6 Variants Comparison

Fit and compare GARCH, GJR-GARCH, and EGARCH models with Normal and Student-t distributions.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from arch import arch_model
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.config import DATA_START_DATE, DATA_END_DATE, TICKER
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load data
price_data = download_data(ticker=TICKER, start=DATA_START_DATE, end=DATA_END_DATE)
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns().dropna() * 100  # Convert to %

print(f"Data loaded: {len(returns)} observations")

In [ ]:
# Define 6 GARCH models
models_config = [
    {'name': 'GARCH(1,1)-Normal', 'vol': 'Garch', 'p': 1, 'q': 1, 'o': 0, 'dist': 'Normal'},
    {'name': 'GARCH(1,1)-Student-t', 'vol': 'Garch', 'p': 1, 'q': 1, 'o': 0, 'dist': 't'},
    {'name': 'GJR-GARCH(1,1)-Normal', 'vol': 'Garch', 'p': 1, 'q': 1, 'o': 1, 'dist': 'Normal'},
    {'name': 'GJR-GARCH(1,1)-Student-t', 'vol': 'Garch', 'p': 1, 'q': 1, 'o': 1, 'dist': 't'},
    {'name': 'EGARCH(1,1)-Normal', 'vol': 'EGARCH', 'p': 1, 'q': 1, 'o': 0, 'dist': 'Normal'},
    {'name': 'EGARCH(1,1)-Student-t', 'vol': 'EGARCH', 'p': 1, 'q': 1, 'o': 0, 'dist': 't'},
]

results = []

for config in models_config:
    try:
        # Fit model
        model = arch_model(
            returns,
            vol=config['vol'],
            p=config['p'],
            o=config['o'],
            q=config['q'],
            mean='Constant',
            rescale=False
        )
        
        model.distribution = config['dist']
        res = model.fit(disp='off')
        
        results.append({
            'Model': config['name'],
            'AIC': res.aic,
            'BIC': res.bic,
            'LL': res.loglikelihood,
        })
        
        logger.info(f"{config['name']}: AIC={res.aic:.2f}")
    except Exception as e:
        logger.error(f"Failed to fit {config['name']}: {e}")

# Display results
results_df = pd.DataFrame(results).sort_values('AIC')
print("\nModel Comparison (sorted by AIC):")
print(results_df.to_string(index=False))

In [ ]:
# Fit best model
best_config = models_config[3]  # GJR-GARCH(1,1)-Student-t

best_model = arch_model(
    returns,
    vol=best_config['vol'],
    p=best_config['p'],
    o=best_config['o'],
    q=best_config['q'],
    mean='Constant'
)
best_model.distribution = best_config['dist']
best_res = best_model.fit(disp='off')

print(f"Best Model: {best_config['name']}")
print(best_res.summary())

In [ ]:
# Plot conditional volatility
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Returns
axes[0].plot(returns.index, returns.values)
axes[0].set_title('Log Returns')
axes[0].set_ylabel('Return (%)')

# Conditional volatility
conditional_vol = best_res.conditional_volatility
axes[1].plot(conditional_vol.index, conditional_vol.values)
axes[1].set_title(f'Conditional Volatility - {best_config["name"]}')
axes[1].set_ylabel('Volatility (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('../report/figures/03_garch_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info("GARCH modeling complete")